# US Revenue Forecast v10.1 — UPSERT 패치

**v10 대비 변경**: `save_to_db` 를 INSERT-only → UPSERT 로 교체.
Override 적용 후 같은 forecast_date 로 재실행 시 DB 의 value 가 자동 갱신됨.

분기 매출 시계열 예측 파이프라인 (FMP API + 시계열 앙상블 + DB 저장).

## 흐름 요약

| Phase | 셀 | 역할 |
|-------|----|------|
| Setup | Cell 1~4 | 환경, 모듈, 파라미터, DB |
| Data  | Cell 5 | **매출 Quality 검증 + Override 통합 함수** |
| Inspect | Cell 6 | 단일 ticker 데이터 정제 결과 확인 |
| Forecast Lib | Cell 7 | 예측 함수 정의 (모델 + index) |
| Storage Lib | Cell 8 | Long-format 변환 + DB 저장 함수 |
| Single Run | Cell 9 | 단일 ticker end-to-end (Quality → 예측 → 저장) |
| Batch | Cell 10 | 전체 배치 (Override 종목 처리 옵션) |
| Monitor | Cell 11 | 저장 결과 조회 |
| Utility | Cell 12 | 오염 데이터 삭제 |

## v8 대비 변경 사항 (v9)

- ❌ v8.1 → v8.2 wrapper chain 제거
- ✅ 단일 `fetch_financial_series` 안에 FPI safety + Override 일체화
- ✅ `MANUAL_REVENUE_OVERRIDES` 운영 dict 한 곳에 집중
- ✅ `OVERRIDE_TICKERS` 자동 도출 + 배치 모드 (all / exclude / only)
- ✅ Cell 5 에서 verification 테스트 분리 → Cell 6 으로 이동


## Cell 1 · 환경 설정 & 경로 자동 감지

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── 후보 프로젝트 루트 (노트북 / 데스크탑) ───────────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]


def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()

    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root

    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "DATA")):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root 후보 경로 : {candidate}")
            return candidate

    raise EnvironmentError("DATA 폴더를 찾을 수 없습니다.")


_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트  : {_ROOT}")
print(f"[확인] DATA 경로     : {os.path.join(_ROOT, 'DATA')}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트  : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] DATA 경로     : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA


## Cell 2 · 모듈 Import

In [2]:
# ── 내부 모듈 (DATA 폴더) ─────────────────────────────────────
from DATA.config import get_db_info, get_engine
from DATA.us_target_ticker_list_2000 import ticker_list as DEFAULT_TICKER_LIST
from DATA.universal_ts_forecast_function_v2 import (
    forecast_sarima, forecast_ets, forecast_prophet,
    forecast_lstm, forecast_theta,
    infer_freq_alias, seasonal_periods_from_freq, clear_memory,
)

# ── 외부 라이브러리 ───────────────────────────────────────────
import gc
import time as _time
import traceback
import requests as _requests
from typing import Optional, List, Dict
from datetime import datetime

import numpy as np
import pandas as pd
from sqlalchemy import text
from IPython.display import display

# ── 로그 유틸 ─────────────────────────────────────────────────
try:
    from DATA.config import log
except ImportError:
    def log(tag: str, msg: str):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}][{tag}] {msg}")


print(f"[OK] 모듈 Import 완료")
print(f"[OK] DEFAULT_TICKER_LIST 길이: {len(DEFAULT_TICKER_LIST):,}개")


[OK] 모듈 Import 완료
[OK] DEFAULT_TICKER_LIST 길이: 2,000개


## Cell 3 · 파라미터 설정

필요 시 이 셀만 수정하세요.

In [3]:
# ════════════════════════════════════════════════════════════
#  ★ 파라미터 — 필요에 따라 이 셀만 수정 ★
# ════════════════════════════════════════════════════════════

# ── FMP API ──────────────────────────────────────────────
FMP_API_KEY    = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"
FMP_BASE_URL   = "https://financialmodelingprep.com/api/v3"
FMP_MAX_RETRY  = 3
FMP_SLEEP_SEC  = 0.35

# ── DB 테이블 ────────────────────────────────────────────
DEST_TABLE = "us_revenue_forecast_data"

# ── 재무 항목 ────────────────────────────────────────────
ITEM       = "sale"

# ── 예측 설정 ────────────────────────────────────────────
HORIZON    = 8     # 예측 분기 수 (8 = 2년)
MIN_OBS    = 28    # 최소 관측 분기 (28 = 7년)

# ── 모델 ─────────────────────────────────────────────────
ALL_MODELS      = ["SARIMA", "ETS", "Prophet", "LSTM", "Theta"]
ENSEMBLE_MODELS = ["SARIMA", "ETS", "Theta"]

# ── 예측 실행일 ──────────────────────────────────────────
FORECAST_DATE = datetime.now().strftime("%Y-%m-%d")

print("[파라미터 확인]")
print(f"  DEST_TABLE   = {DEST_TABLE}")
print(f"  ITEM         = {ITEM}")
print(f"  HORIZON      = {HORIZON}분기")
print(f"  MIN_OBS      = {MIN_OBS}개")
print(f"  ENSEMBLE     = {ENSEMBLE_MODELS}")
print(f"  FORECAST_DATE= {FORECAST_DATE}")


[파라미터 확인]
  DEST_TABLE   = us_revenue_forecast_data
  ITEM         = sale
  HORIZON      = 8분기
  MIN_OBS      = 28개
  ENSEMBLE     = ['SARIMA', 'ETS', 'Theta']
  FORECAST_DATE= 2026-05-07


## Cell 4 · DB 연결 테스트

In [4]:
db_info = get_db_info()
engine  = get_engine(db_info)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("[OK] DB 연결 성공")
    print(f"     host={db_info.get('host')}  port={db_info.get('port')}  db={db_info.get('database')}")
except Exception as e:
    print(f"[FAIL] DB 연결 실패: {e}")


[OK] DB 연결 성공
     host=192.168.0.230  port=3307  db=investar


## Cell 5 · 매출 데이터 Quality 검증 + Manual Override (통합)

**v9 정리**: 이전 v8.1 / v8.2 wrapper chain 을 모두 단일 셀로 통합했습니다.

### 5-1. Helper 함수
- `_fmp_fetch_income` — FMP /income-statement 1차 조회
- `_fmp_fetch_income_as_reported` — XBRL 백업 endpoint
- `_fmp_fetch_profile` — country/isAdr 으로 FPI 식별
- `_clean_series` — 중복 제거 정제
- `_detect_duplicate_value_anomaly` — 마지막 분기 ≡ 직전 분기 패턴 탐지
- `_apply_manual_overrides` — 알려진 정답 주입

### 5-2. `MANUAL_REVENUE_OVERRIDES`
1차 출처(SEC 6-K 등)에서 직접 확인한 정답 분기 매출 등록.
**FMP 가 정상화되면 verbose 로그에 "제거 가능" 알림이 뜸**.

### 5-3. `fetch_financial_series` (단일 통합 함수)
파이프라인:
1. FMP primary 조회
2. 음수 매출 검사
3. 중복값 의심 탐지 → 의심 시 as-reported XBRL fallback → 그래도 의심이면 strict drop
4. Manual override 적용 (drop 된 분기 복원 또는 잘못된 값 갱신)
5. 최종 min_obs 검증

### 5-4. Override 종목 조회 도구
`OVERRIDE_TICKERS` (set), `get_override_summary()` (df 형태).

In [5]:
# ════════════════════════════════════════════════════════════════════
# 5-1. Helper 함수 정의
# ════════════════════════════════════════════════════════════════════

def _fmp_fetch_income(ticker: str, limit: int = 40,
                      period: str = "quarter") -> pd.DataFrame:
    """FMP /income-statement → 표준 columns DataFrame.
    columns = [date, report_date, period, date_month, value]
    """
    url = f"{FMP_BASE_URL}/income-statement/{ticker}"
    params = {"period": period, "limit": limit, "apikey": FMP_API_KEY}

    for k in range(FMP_MAX_RETRY):
        try:
            r = _requests.get(url, params=params, timeout=30)
            if r.status_code == 429:
                _time.sleep(1.5 + k); continue
            r.raise_for_status()
            data = r.json()
            if isinstance(data, dict) and "Error Message" in data:
                raise ValueError(data["Error Message"])
            break
        except Exception as e:
            if k == FMP_MAX_RETRY - 1:
                raise RuntimeError(f"FMP 조회 실패 ({ticker}): {e}")
            _time.sleep(FMP_SLEEP_SEC + k * 0.5)

    if not data or not isinstance(data, list):
        return pd.DataFrame()

    df = pd.DataFrame(data)
    if df.empty or "revenue" not in df.columns:
        return pd.DataFrame()

    df["date"]        = pd.to_datetime(df["date"], errors="coerce")
    df["report_date"] = pd.to_datetime(
        df.get("fillingDate", df.get("acceptedDate", pd.NaT)), errors="coerce")
    df["period"]      = df.get("period", pd.NA)
    df["date_month"]  = df["date"].dt.to_period("M").dt.to_timestamp()
    df["value"]       = pd.to_numeric(df["revenue"], errors="coerce")

    return (df[["date","report_date","period","date_month","value"]]
                .dropna(subset=["date","value"])
                .sort_values("date")
                .reset_index(drop=True))


def _fmp_fetch_income_as_reported(ticker: str, limit: int = 40,
                                    period: str = "quarter") -> pd.DataFrame:
    """FMP /income-statement-as-reported (XBRL) — fallback 전용."""
    url = f"{FMP_BASE_URL}/income-statement-as-reported/{ticker}"
    params = {"period": period, "limit": limit, "apikey": FMP_API_KEY}

    for k in range(FMP_MAX_RETRY):
        try:
            r = _requests.get(url, params=params, timeout=30)
            if r.status_code == 429:
                _time.sleep(1.5 + k); continue
            r.raise_for_status()
            data = r.json()
            if isinstance(data, dict) and "Error Message" in data:
                return pd.DataFrame()
            break
        except Exception:
            if k == FMP_MAX_RETRY - 1:
                return pd.DataFrame()
            _time.sleep(FMP_SLEEP_SEC + k * 0.5)

    if not data or not isinstance(data, list):
        return pd.DataFrame()

    df = pd.DataFrame(data)
    if df.empty:
        return pd.DataFrame()

    REVENUE_TAGS = [
        "revenues",
        "revenuefromcontractwithcustomerexcludingassessedtax",
        "revenuefromcontractwithcustomerincludingassessedtax",
        "salesrevenuenet", "salesrevenuegoodsnet", "revenue",
    ]
    rev_col = next((tag for tag in REVENUE_TAGS if tag in df.columns), None)
    if rev_col is None:
        return pd.DataFrame()

    df["date"]        = pd.to_datetime(df.get("date"), errors="coerce")
    df["report_date"] = pd.to_datetime(
        df.get("fillingDate", df.get("acceptedDate", pd.NaT)), errors="coerce")
    df["period"]      = df.get("period", pd.NA)
    df["date_month"]  = df["date"].dt.to_period("M").dt.to_timestamp()
    df["value"]       = pd.to_numeric(df[rev_col], errors="coerce")

    return (df[["date","report_date","period","date_month","value"]]
                .dropna(subset=["date","value"])
                .sort_values("date")
                .reset_index(drop=True))


def _fmp_fetch_profile(ticker: str) -> Dict:
    """FMP /profile → country, isAdr, is_fpi (heuristic)."""
    url = f"{FMP_BASE_URL}/profile/{ticker}"
    try:
        r = _requests.get(url, params={"apikey": FMP_API_KEY}, timeout=15)
        r.raise_for_status()
        data = r.json()
        if isinstance(data, list) and data:
            p = data[0]
            country = (p.get("country") or "").upper()
            is_adr  = bool(p.get("isAdr", False))
            is_fpi  = is_adr or (country not in ("US", ""))
            return {"country": country, "is_adr": is_adr, "is_fpi": is_fpi,
                    "industry": p.get("industry", "") or ""}
    except Exception:
        pass
    return {"country": "", "is_adr": False, "is_fpi": False, "industry": ""}


def _clean_series(df: pd.DataFrame) -> pd.DataFrame:
    """date 중복 제거 (마지막 행 유지)."""
    df = df.copy()
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["date","value"]).sort_values("date").reset_index(drop=True)
    return df.groupby("date", sort=True).last().reset_index()


def _detect_duplicate_value_anomaly(df: pd.DataFrame,
                                     rel_tol: float = 1e-6,
                                     abs_tol: float = 1.0) -> Dict:
    """마지막 분기 == 직전 분기 → 의심 신호 (FMP carry-forward 패턴)."""
    if df is None or len(df) < 2:
        return {"anomaly": False, "reason": "데이터 부족"}

    last, prev = df.iloc[-1], df.iloc[-2]
    lv, pv = float(last["value"]), float(prev["value"])

    if pv == 0:
        return {"anomaly": False, "reason": "직전값=0"}

    diff_abs = abs(lv - pv)
    diff_rel = diff_abs / max(abs(pv), 1)
    is_dup   = (diff_abs <= abs_tol) and (diff_rel <= rel_tol)

    return {
        "anomaly":    is_dup,
        "last_date":  str(last["date"].date()) if pd.notna(last["date"]) else None,
        "prev_date":  str(prev["date"].date()) if pd.notna(prev["date"]) else None,
        "last_value": lv, "prev_value": pv,
        "reason":     f"마지막={lv:,.0f} ≡ 직전={pv:,.0f}" if is_dup
                      else f"Δ={diff_abs:,.0f} ({diff_rel:.4%})",
    }


# ════════════════════════════════════════════════════════════════════
# 5-2. MANUAL_REVENUE_OVERRIDES — 1차 출처 정답 등록
# ════════════════════════════════════════════════════════════════════
#
# 형식: ticker → {"YYYY-MM-DD" (분기말): value (USD)}
#
# 등록 기준:
#   · SEC 6-K, 20-F, 회사 IR 페이지 등 1차 출처에서 직접 확인한 값만
#   · 분기말 날짜는 calendar Q-end (3/31, 6/30, 9/30, 12/31)
#   · 값은 USD 정수 (예: 342_100_000)
#
# 정리 시점:
#   · FMP 가 데이터 수정 후 verbose 로그에 "제거 가능 ✓" 알림 → 그때 dict 에서 제거
# ════════════════════════════════════════════════════════════════════

MANUAL_REVENUE_OVERRIDES: Dict[str, Dict[str, float]] = {
    "SIMO": {
        "2026-03-31": 342_100_000,   # Q1 2026, 출처: 4/28 6-K (filing 2026-04-29)
                                     # FMP 가 Q4 2025 값 ($278.5M) 으로 carry-forward
    },
    # ── 신규 문제 종목 추가 시 여기에 ──────────────────────────────
    # "TICKER": {"YYYY-MM-DD": value_in_usd},
}


def _apply_manual_overrides(df: pd.DataFrame, ticker: str,
                             verbose: bool = False) -> pd.DataFrame:
    """알려진 정답 값으로 override / drop 된 분기 보강."""
    overrides = MANUAL_REVENUE_OVERRIDES.get(ticker.upper())
    if not overrides:
        return df

    df = df.copy()
    for date_str, true_value in overrides.items():
        target_date = pd.Timestamp(date_str)
        mask = df["date"] == target_date

        if mask.any():
            current  = float(df.loc[mask, "value"].iloc[0])
            rel_diff = abs(current - true_value) / max(abs(true_value), 1)
            if rel_diff > 0.01:
                df.loc[mask, "value"] = float(true_value)
                if verbose:
                    print(f"  [override] {ticker} {date_str}: "
                          f"{current:,.0f} → {true_value:,.0f}")
            else:
                if verbose:
                    print(f"  [override-match] {ticker} {date_str}: "
                          f"FMP 값과 일치 ({true_value:,.0f}) "
                          f"— MANUAL_REVENUE_OVERRIDES 에서 제거 가능 ✓")
        else:
            new_row = pd.DataFrame([{
                "date":        target_date,
                "report_date": pd.NaT,
                "period":      f"Q{target_date.quarter}",
                "date_month":  target_date.to_period("M").to_timestamp(),
                "value":       float(true_value),
            }])
            df = pd.concat([df, new_row], ignore_index=True)
            df = df.sort_values("date").reset_index(drop=True)
            if verbose:
                print(f"  [override-add] {ticker} {date_str}: "
                      f"+ {true_value:,.0f} (drop 분기 복원)")

    return df


# ════════════════════════════════════════════════════════════════════
# 5-3. fetch_financial_series — 통합 함수 (Quality + Override)
# ════════════════════════════════════════════════════════════════════

def fetch_financial_series(
    engine,
    ticker: str,
    item: str = "sale",
    min_obs: int = 28,
    *,
    strict_dup_check: bool        = True,
    fallback_to_as_reported: bool = True,
    verbose: bool                 = False,
) -> pd.DataFrame:
    """FMP 분기 매출 시계열 추출 (v9 통합).

    Pipeline:
      1) FMP primary 조회
      2) 음수 매출 검사
      3) 중복값 의심 탐지 → fallback (XBRL) 시도
      4) Strict drop (의심값 제거)
      5) Manual override 적용 (정답 주입)
      6) 최종 min_obs 검증

    Args:
        strict_dup_check : 의심값 fallback 도 실패 시 drop 여부
        fallback_to_as_reported : XBRL endpoint 시도 여부
        verbose : 진단 로그 출력
    """
    # ── Step 1: Primary fetch ───────────────────────────────────
    df = _fmp_fetch_income(ticker, limit=max(min_obs + 10, 40))
    if df.empty:
        raise ValueError(f"[{ticker}] FMP 1차 조회 결과 없음")
    _time.sleep(FMP_SLEEP_SEC)
    df = _clean_series(df)

    # ── Step 2: 음수 매출 검사 ──────────────────────────────────
    if (df["value"] < 0).any():
        neg_dates = df.loc[df["value"] < 0, "date"].dt.date.tolist()
        raise ValueError(f"[{ticker}] 음수 매출 → 예측 제외 ({neg_dates})")

    # ── Step 3+4: 중복값 의심 → Fallback → Strict drop ─────────
    diag = _detect_duplicate_value_anomaly(df)
    if diag["anomaly"]:
        prof = _fmp_fetch_profile(ticker)
        _time.sleep(FMP_SLEEP_SEC)

        if verbose:
            fpi_tag = " [FPI]" if prof["is_fpi"] else ""
            print(f"  ⚠ [{ticker}]{fpi_tag} 중복값 의심: "
                  f"{diag['prev_date']}={diag['prev_value']:,.0f} ≡ "
                  f"{diag['last_date']}={diag['last_value']:,.0f} "
                  f"(country={prof['country']}, isAdr={prof['is_adr']})")

        used_fallback = False
        if fallback_to_as_reported:
            df_ar = _fmp_fetch_income_as_reported(ticker, limit=max(min_obs + 10, 40))
            _time.sleep(FMP_SLEEP_SEC)
            if not df_ar.empty:
                df_ar = _clean_series(df_ar)
                diag_ar = _detect_duplicate_value_anomaly(df_ar)
                if not diag_ar["anomaly"] and len(df_ar) >= len(df) - 1:
                    if verbose:
                        last = df_ar.iloc[-1]
                        print(f"  ✓ [{ticker}] as-reported (XBRL) fallback 성공: "
                              f"{last['date'].date()}={last['value']:,.0f}")
                    df = df_ar
                    used_fallback = True
                elif verbose:
                    print(f"  ✗ [{ticker}] as-reported 도 의심: {diag_ar['reason']}")

        if (not used_fallback) and strict_dup_check:
            dropped = df.iloc[-1]
            df = df.iloc[:-1].reset_index(drop=True)
            if verbose:
                print(f"  → [{ticker}] strict drop "
                      f"({dropped['date'].date()}={dropped['value']:,.0f}) "
                      f"→ 잔여 {len(df)}분기")

    # ── Step 5: Manual override 적용 ────────────────────────────
    df = _apply_manual_overrides(df, ticker, verbose=verbose)

    # ── Step 6: 최종 min_obs 검증 ───────────────────────────────
    if len(df) < min_obs:
        raise ValueError(f"[{ticker}] 관측치 부족: {len(df)} < {min_obs}")

    return df


# ════════════════════════════════════════════════════════════════════
# 5-4. Override 종목 조회 도구
# ════════════════════════════════════════════════════════════════════

# 자동으로 도출되는 override 종목 set (대문자 정규화)
OVERRIDE_TICKERS = {tk.upper() for tk in MANUAL_REVENUE_OVERRIDES.keys()}


def get_override_summary() -> pd.DataFrame:
    """MANUAL_REVENUE_OVERRIDES 를 DataFrame 으로 보기 좋게 출력.

    Columns: ticker, quarter_end, override_value, override_value_M
    """
    rows = []
    for tk, qs in MANUAL_REVENUE_OVERRIDES.items():
        for d, v in qs.items():
            rows.append({
                "ticker":           tk,
                "quarter_end":      d,
                "override_value":   v,
                "override_value_M": f"${v/1e6:,.1f}M",
            })
    if not rows:
        return pd.DataFrame(columns=["ticker","quarter_end","override_value","override_value_M"])
    return pd.DataFrame(rows).sort_values(["ticker","quarter_end"]).reset_index(drop=True)


# ════════════════════════════════════════════════════════════════════
# 5-5. sync_overrides_to_db — DB 의 actual 값을 override 정답으로 강제 동기화
# ════════════════════════════════════════════════════════════════════
#
# 사용 시나리오:
#   override 추가 후, 이미 DB 에 옛 값(예: $278.5M)이 저장된 상태에서
#   해당 ticker 를 재예측 했지만 unique key 충돌로 갱신이 안 된 경우.
#   이 함수를 한 번 호출하면 모든 override 항목을 DB 에 강제 반영함.
#
# 사용:
#   sync_overrides_to_db(engine)
#
# 운영 워크플로우:
#   1. MANUAL_REVENUE_OVERRIDES 에 항목 추가
#   2. Cell 5 재실행 (dict 갱신)
#   3. sync_overrides_to_db(engine) 한 번 실행
#   4. FCFF / Relative_Valuation 등 downstream 노트북 재실행
# ════════════════════════════════════════════════════════════════════

def sync_overrides_to_db(engine):
    """MANUAL_REVENUE_OVERRIDES 의 모든 값을 DB 의 actual 행에 강제 반영.

    각 (ticker, date) 의 모든 forecast_date 에 걸쳐 value 를 갱신.
    DB 에 해당 행이 없으면 갱신 없음 (해당 ticker 매출 예측을 먼저 실행 필요).

    Returns:
        int : 총 갱신된 행 수
    """
    if not MANUAL_REVENUE_OVERRIDES:
        print("[INFO] MANUAL_REVENUE_OVERRIDES 가 비어있음 — 갱신 없음")
        return 0

    total_updated = 0
    with engine.begin() as conn:
        for ticker, qs in MANUAL_REVENUE_OVERRIDES.items():
            for date_str, value in qs.items():
                r = conn.execute(text(f"""
                    UPDATE `{DEST_TABLE}`
                    SET    value = :v
                    WHERE  ticker = :tk
                      AND  item = 'sale'
                      AND  data_type = 'actual'
                      AND  date = :d
                """), {"v": int(value), "tk": ticker, "d": date_str})
                n = r.rowcount or 0
                total_updated += n
                if n == 0:
                    print(f"  [SKIP]   {ticker} {date_str} → DB 에 해당 행 없음 "
                          f"(매출 예측 먼저 실행 필요)")
                else:
                    print(f"  [UPDATE] {ticker} {date_str} → ${value/1e6:,.1f}M  ({n}행 갱신)")

    print(f"\n[OK] 총 {total_updated}행 갱신")
    return total_updated


# ════════════════════════════════════════════════════════════════════
# 적용 확인
# ════════════════════════════════════════════════════════════════════
print("[OK] Cell 5 — fetch_financial_series + Override 통합 (v9)")
print(f"     · 등록된 OVERRIDE_TICKERS: {len(OVERRIDE_TICKERS)}개 → {sorted(OVERRIDE_TICKERS)}")
print(f"     · Pipeline: FMP → as-reported → strict drop → override → min_obs check")


[OK] Cell 5 — fetch_financial_series + Override 통합 (v9)
     · 등록된 OVERRIDE_TICKERS: 1개 → ['SIMO']
     · Pipeline: FMP → as-reported → strict drop → override → min_obs check


## Cell 6 · 단일 Ticker 매출 데이터 Quality 검증

`TEST_TICKER` 만 수정해서 사용. 다음을 확인합니다:

1. FMP raw 데이터에서 **중복값 의심** 자동 탐지 여부
2. **Override 적용**된 최종 분기 (해당 종목이 등록되어 있을 경우)
3. 최종 정제된 시계열 (예측 입력으로 그대로 사용 가능)

In [6]:
# ─── 단일 ticker 입력 ────────────────────────────────────────
TEST_TICKER = "SIMO"     # ← 검증할 ticker (다른 종목으로 변경 가능)

print(f"{'='*72}")
print(f"[Quality Check] {TEST_TICKER}")
print(f"{'='*72}")

# Override 등록 여부 표시
if TEST_TICKER.upper() in OVERRIDE_TICKERS:
    overrides = MANUAL_REVENUE_OVERRIDES[TEST_TICKER.upper()]
    print(f"  ※ 이 ticker 는 override 등록되어 있음:")
    for d, v in overrides.items():
        print(f"     - {d}: ${v/1e6:,.1f}M")
    print()

# 통합 fetch 호출 (verbose 로 진단 로그 표시)
try:
    src_df = fetch_financial_series(
        engine, TEST_TICKER, item=ITEM, min_obs=MIN_OBS,
        strict_dup_check=True, fallback_to_as_reported=True, verbose=True,
    )

    print(f"\n[OK] {TEST_TICKER} 정제 완료: {len(src_df)}분기")
    print(f"     기간: {src_df['date'].iloc[0].date()} ~ {src_df['date'].iloc[-1].date()}")
    print(f"\n[정제된 매출 시계열] 마지막 8분기:")
    display(src_df.tail(8))

except Exception as e:
    print(f"\n[FAIL] {e}")
    src_df = None

# ─── Override 종목 전체 목록 표시 (참조용) ────────────────────
print(f"\n[현재 등록된 Override 종목]")
display(get_override_summary())


[Quality Check] SIMO
  ※ 이 ticker 는 override 등록되어 있음:
     - 2026-03-31: $342.1M

  [override-match] SIMO 2026-03-31: FMP 값과 일치 (342,100,000) — MANUAL_REVENUE_OVERRIDES 에서 제거 가능 ✓

[OK] SIMO 정제 완료: 40분기
     기간: 2016-06-30 ~ 2026-03-31

[정제된 매출 시계열] 마지막 8분기:


,date,report_date,period,date_month,value
32,2024-06-30,2024-08-02,Q2,2024-06-01,210670000
33,2024-09-30,2024-10-31,Q3,2024-09-01,212412000
34,2024-12-31,2025-04-30,Q4,2024-12-01,191160000
35,2025-03-30,2025-04-30,Q1,2025-03-01,166492000
36,2025-06-30,2025-07-31,Q2,2025-06-01,198675000
37,2025-09-30,2025-10-31,Q3,2025-09-01,241999000
38,2025-12-31,2026-04-30,Q4,2025-12-01,278461000
39,2026-03-31,2026-04-29,Q1,2026-03-01,342105000



[현재 등록된 Override 종목]


,ticker,quarter_end,override_value,override_value_M
0,SIMO,2026-03-31,342100000,$342.1M


## Cell 7 · 예측 함수 정의

- `forecast_one_ticker` — SARIMA / ETS / Prophet / LSTM / Theta 호출
- `make_forecast_index_v5` — 분기 예측 인덱스 생성 (단순 +3 month, fiscal calendar 호환)

In [7]:
def forecast_one_ticker(
    y: pd.Series,
    ticker: str,
    horizon: int,
    models: list,
) -> dict:
    """단일 ticker 시계열 → 지정 모델로 예측."""
    import psutil, os
    proc = psutil.Process(os.getpid())

    def _mem_mb():
        return proc.memory_info().rss / 1024 / 1024

    freq    = infer_freq_alias(y.index)
    sp      = seasonal_periods_from_freq(freq)
    results = {}

    def _call(model_name):
        if model_name == "SARIMA":
            return forecast_sarima(y, horizon, seasonal_period=sp)
        elif model_name == "ETS":
            return forecast_ets(y, horizon, m=sp)
        elif model_name == "Prophet":
            return forecast_prophet(y, horizon, m=sp)
        elif model_name == "LSTM":
            return forecast_lstm(y, horizon)
        elif model_name == "Theta":
            return forecast_theta(y, horizon, m=sp)
        else:
            raise ValueError(f"알 수 없는 모델: {model_name}")

    for model_name in models:
        log(ticker, f"  [{model_name}] 시작  (메모리: {_mem_mb():.1f} MB)")
        try:
            res = _call(model_name)
            results[model_name] = res
            fc_arr = np.asarray(res.get("forecast", []))
            if len(fc_arr) > 0:
                log(ticker, f"  [{model_name}] 완료  첫값={fc_arr[0]:.2e} 마지막={fc_arr[-1]:.2e}")
            else:
                log(ticker, f"  [{model_name}] 완료 (forecast 비어있음)")
        except Exception as e:
            log(ticker, f"  [{model_name}] 실패: {type(e).__name__}: {str(e)[:80]}")
            results[model_name] = {"error": f"{type(e).__name__}: {e}"}

    return results


# ════════════════════════════════════════════════════════════════════
# 분기 예측 인덱스 — calendar month-end +3
# ════════════════════════════════════════════════════════════════════
def make_forecast_index_v5(last_date, horizon, freq=None):
    """last_date 의 month + 3 부터 calendar month-end 까지 horizon 개.

    fiscal calendar 변형 (1월/2월/8월/9월 fiscal year, 13W 등) 모두 자연 처리.
    """
    dates = []
    m, y = last_date.month, last_date.year
    for _ in range(horizon):
        m += 3
        if m > 12:
            m -= 12
            y += 1
        dates.append(pd.Timestamp(y, m, 1) + pd.offsets.MonthEnd(0))
    return pd.DatetimeIndex(dates)


def infer_freq_alias_v5(index):
    return "QE"  # placeholder for compatibility


# 모듈 + 노트북 namespace 둘 다 patch
import DATA.universal_ts_forecast_function_v2 as _ufm
_ufm.infer_freq_alias    = infer_freq_alias_v5
_ufm.make_forecast_index = make_forecast_index_v5
infer_freq_alias    = infer_freq_alias_v5
make_forecast_index = make_forecast_index_v5

print("[OK] Cell 7 — forecast_one_ticker + make_forecast_index 정의")


[OK] Cell 7 — forecast_one_ticker + make_forecast_index 정의


## Cell 8 · Long-format 변환 + DB 저장 함수

- `build_long_df` — actual + 모델별 forecast + Ensemble → long 포맷
- `ensure_table` — 저장 테이블 생성 (최초 1회)
- `save_to_db` — `(ticker,item,date,model,forecast_date)` 중복 키 기준 신규만 INSERT

In [8]:
def build_long_df(
    ticker: str,
    item: str,
    src_df: pd.DataFrame,
    forecast_results: dict,
    forecast_index: pd.DatetimeIndex,
    forecast_date: str,
    ensemble_models: list = None,
) -> pd.DataFrame:
    """actual + forecast (모델별) + Ensemble → long-format DataFrame."""
    if ensemble_models is None:
        ensemble_models = ENSEMBLE_MODELS

    now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rows = []

    # actual 행
    for _, row in src_df.iterrows():
        rows.append({
            "ticker": ticker, "item": item,
            "date":   row["date"].strftime("%Y-%m-%d"),
            "period": row.get("period"),
            "date_month": (row["date_month"].strftime("%Y-%m-%d")
                            if pd.notna(row.get("date_month")) else None),
            "data_type": "actual", "model": "actual",
            "value": round(float(row["value"]), 6),
            "forecast_date": forecast_date,
            "sarima_order": None, "sarima_seasonal_order": None,
            "sarima_ic_value": None, "created_at": now_str,
        })

    # forecast 행
    ensemble_bucket = {}
    for model_name, res in forecast_results.items():
        if "error" in res or "forecast" not in res:
            continue
        fc_arr = np.asarray(res["forecast"])
        spec   = res.get("spec", {})

        sarima_order, sarima_seasonal, sarima_ic = None, None, None
        if model_name == "SARIMA":
            sarima_order    = str(spec.get("order", ""))
            sarima_seasonal = str(spec.get("seasonal_order", ""))
            raw_ic = spec.get("ic_value")
            if raw_ic is not None:
                try:
                    v = float(raw_ic)
                    sarima_ic = round(v, 4) if np.isfinite(v) else None
                except (TypeError, ValueError):
                    pass

        for i, dt in enumerate(forecast_index):
            if i >= len(fc_arr):
                break
            val    = float(fc_arr[i])
            dt_str = dt.strftime("%Y-%m-%d")
            rows.append({
                "ticker": ticker, "item": item, "date": dt_str,
                "period": None, "date_month": None,
                "data_type": "forecast", "model": model_name,
                "value": round(val, 6), "forecast_date": forecast_date,
                "sarima_order": sarima_order,
                "sarima_seasonal_order": sarima_seasonal,
                "sarima_ic_value": sarima_ic, "created_at": now_str,
            })
            if model_name in ensemble_models:
                ensemble_bucket.setdefault(dt_str, []).append(val)

    # Ensemble 행
    for dt_str, vals in ensemble_bucket.items():
        rows.append({
            "ticker": ticker, "item": item, "date": dt_str,
            "period": None, "date_month": None,
            "data_type": "forecast", "model": "Ensemble",
            "value": round(float(np.mean(vals)), 6),
            "forecast_date": forecast_date,
            "sarima_order": None, "sarima_seasonal_order": None,
            "sarima_ic_value": None, "created_at": now_str,
        })

    return pd.DataFrame(rows)


# ════════════════════════════════════════════════════════════════════
# DB 테이블 + 저장
# ════════════════════════════════════════════════════════════════════
CREATE_TABLE_SQL = f"""
CREATE TABLE IF NOT EXISTS `{DEST_TABLE}` (
    id                    BIGINT       NOT NULL AUTO_INCREMENT,
    ticker                VARCHAR(20)  NOT NULL,
    item                  VARCHAR(30)  NOT NULL,
    date                  DATE         NOT NULL,
    period                VARCHAR(10)  DEFAULT NULL,
    date_month            DATE         DEFAULT NULL,
    data_type             VARCHAR(10)  NOT NULL COMMENT 'actual / forecast',
    model                 VARCHAR(20)  NOT NULL,
    value                 DOUBLE       DEFAULT NULL,
    forecast_date         DATE         NOT NULL,
    sarima_order          VARCHAR(30)  DEFAULT NULL,
    sarima_seasonal_order VARCHAR(30)  DEFAULT NULL,
    sarima_ic_value       DOUBLE       DEFAULT NULL,
    created_at            DATETIME     DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (id),
    UNIQUE KEY uq_main (ticker, item, date, model, forecast_date),
    INDEX idx_ticker      (ticker),
    INDEX idx_forecast_dt (forecast_date),
    INDEX idx_model       (model)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
"""


def ensure_table(engine):
    with engine.begin() as conn:
        conn.execute(text(CREATE_TABLE_SQL))


def save_to_db(engine, long_df: pd.DataFrame, dest_table: str = DEST_TABLE) -> int:
    """UPSERT 로 저장.

    동작 (v10.1):
      · 새 unique key (ticker,item,date,model,forecast_date) → INSERT
      · 기존 unique key 존재 → value, sarima_*, created_at 갱신
      · override 적용으로 같은 key 의 값이 변경된 경우에도 자동 반영

    Returns:
        affected rows (MySQL: INSERT=1, UPDATE=2, no-change=0 의 합)
    """
    if long_df is None or long_df.empty:
        return 0

    upsert_sql = text(f"""
        INSERT INTO `{dest_table}`
            (ticker, item, date, period, date_month, data_type, model, value,
             forecast_date, sarima_order, sarima_seasonal_order, sarima_ic_value,
             created_at)
        VALUES
            (:ticker, :item, :date, :period, :date_month, :data_type, :model, :value,
             :forecast_date, :sarima_order, :sarima_seasonal_order, :sarima_ic_value,
             :created_at)
        ON DUPLICATE KEY UPDATE
            value                 = VALUES(value),
            period                = VALUES(period),
            date_month            = VALUES(date_month),
            sarima_order          = VALUES(sarima_order),
            sarima_seasonal_order = VALUES(sarima_seasonal_order),
            sarima_ic_value       = VALUES(sarima_ic_value),
            created_at            = VALUES(created_at)
    """)

    # NaN / NaT → None 변환 (SQLAlchemy bind 호환)
    df = long_df.copy()
    df = df.where(pd.notna(df), None)
    records = df.to_dict("records")

    with engine.begin() as conn:
        result = conn.execute(upsert_sql, records)
        affected = result.rowcount or 0

    return affected


print("[OK] Cell 8 — build_long_df / ensure_table / save_to_db 정의")


[OK] Cell 8 — build_long_df / ensure_table / save_to_db 정의


## Cell 9 · 단일 Ticker 매출 예측 → DB 저장 (end-to-end)

호영님 요청 4·6 — 한 셀에서 데이터 정제 → 예측 → DB 저장까지 일괄 수행.

`SINGLE_TICKER` 만 수정하시면 됩니다.

In [9]:
# ─── 입력 ────────────────────────────────────────────────────
SINGLE_TICKER = "SIMO"           # ← 실행할 ticker
SINGLE_MODELS = ALL_MODELS       # ← 또는 ["SARIMA","ETS","Theta"] (빠른 실행)
SAVE_TO_DB    = True             # ← False 로 두면 dry-run

print(f"{'='*72}")
print(f"[Single Ticker End-to-End] {SINGLE_TICKER}")
print(f"  Models: {SINGLE_MODELS}")
print(f"  Save to DB: {SAVE_TO_DB}")
print(f"{'='*72}\n")

# ─── Step 1: 매출 데이터 정제 (Quality + Override 통합) ──────
print("[Step 1] 매출 데이터 정제 ...")
src_df = fetch_financial_series(
    engine, SINGLE_TICKER, item=ITEM, min_obs=MIN_OBS,
    strict_dup_check=True, fallback_to_as_reported=True, verbose=True,
)
print(f"  → {len(src_df)} 분기 정제 완료\n")

# ─── Step 2: 시계열 변환 ─────────────────────────────────────
y = src_df.set_index("date")["value"].copy()
y.index = pd.DatetimeIndex(y.index)
y.name  = ITEM

# ─── Step 3: 예측 ────────────────────────────────────────────
print(f"[Step 2] 예측 ({SINGLE_MODELS}) ...")
fc_results = forecast_one_ticker(y, SINGLE_TICKER, HORIZON, SINGLE_MODELS)

# ─── Step 4: 예측 인덱스 ─────────────────────────────────────
fc_index = make_forecast_index(y.index[-1], HORIZON)

print(f"\n[예측 결과 요약]")
for m, res in fc_results.items():
    if "error" in res:
        print(f"  {m:<10}: ERROR → {res['error'][:80]}")
    else:
        fc = np.asarray(res["forecast"])
        print(f"  {m:<10}: 첫값={fc[0]/1e6:.1f}M  마지막={fc[-1]/1e6:.1f}M")

# ─── Step 5: Long-format 변환 ────────────────────────────────
long_df = build_long_df(
    ticker=SINGLE_TICKER, item=ITEM, src_df=src_df,
    forecast_results=fc_results, forecast_index=fc_index,
    forecast_date=FORECAST_DATE,
)
print(f"\n[Step 3] Long-format: {len(long_df)} 행 (actual + forecast + Ensemble)")

# ─── Step 6: DB 저장 ─────────────────────────────────────────
if SAVE_TO_DB:
    ensure_table(engine)
    affected = save_to_db(engine, long_df)
    print(f"\n[Step 4] DB 저장 완료 — affected rows: {affected} (input: {len(long_df)})")
    print(f"     ※ MySQL: INSERT=1, UPDATE=2, no-change=0 의 합. >0 이면 DB 변경 발생.")
else:
    print(f"\n[Step 4] dry-run mode — DB 저장 스킵")

# ─── 검증 표시 ───────────────────────────────────────────────
print(f"\n{'='*72}")
print(f"[검증] 마지막 actual 분기 + 첫 forecast 분기")
print(f"{'='*72}")
sample = long_df[long_df["model"].isin(["actual","Ensemble","SARIMA"])].tail(20)
display(sample[["date","data_type","model","value"]])


[Single Ticker End-to-End] SIMO
  Models: ['SARIMA', 'ETS', 'Prophet', 'LSTM', 'Theta']
  Save to DB: True

[Step 1] 매출 데이터 정제 ...
  [override-match] SIMO 2026-03-31: FMP 값과 일치 (342,100,000) — MANUAL_REVENUE_OVERRIDES 에서 제거 가능 ✓
  → 40 분기 정제 완료

[Step 2] 예측 (['SARIMA', 'ETS', 'Prophet', 'LSTM', 'Theta']) ...
[SIMO]   [SARIMA] 시작  (메모리: 435.0 MB)
[메모리] forecast_sarima 실행 전: 435.01 MB
[메모리] find_best_sarima_params 실행 전: 435.03 MB
[메모리] find_best_sarima_params 실행 후: 446.14 MB (변화: +11.11 MB)
[메모리] forecast_sarima 실행 후: 446.23 MB (변화: +11.22 MB)
[SIMO]   [SARIMA] 완료  첫값=3.62e+08 마지막=3.70e+08
[SIMO]   [ETS] 시작  (메모리: 446.2 MB)
[메모리] forecast_ets 실행 전: 446.23 MB
[메모리] forecast_ets 실행 후: 446.52 MB (변화: +0.28 MB)
[SIMO]   [ETS] 완료  첫값=5.41e+08 마지막=7.56e+09
[SIMO]   [Prophet] 시작  (메모리: 446.5 MB)
[메모리] forecast_prophet 실행 전: 446.52 MB


22:05:35 - cmdstanpy - INFO - Chain [1] start processing
22:05:35 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 447.95 MB (변화: +1.43 MB)
[SIMO]   [Prophet] 완료  첫값=5.29e+09 마지막=6.79e+09
[SIMO]   [LSTM] 시작  (메모리: 447.9 MB)
[메모리] forecast_lstm 실행 전: 447.95 MB
[메모리] forecast_lstm 실행 후: 1571.33 MB (변화: +1123.39 MB)
[경고] 메모리 사용량이 크게 증가했습니다. 메모리 정리를 권장합니다.
[SIMO]   [LSTM] 완료  첫값=2.32e+08 마지막=2.76e+08
[SIMO]   [Theta] 시작  (메모리: 1571.3 MB)
[메모리] forecast_theta 실행 전: 1571.33 MB
[메모리] forecast_theta 실행 후: 1571.49 MB (변화: +0.16 MB)
[SIMO]   [Theta] 완료  첫값=3.45e+08 마지막=3.68e+08

[예측 결과 요약]
  SARIMA    : 첫값=362.0M  마지막=369.8M
  ETS       : 첫값=541.0M  마지막=7560.2M
  Prophet   : 첫값=5290.7M  마지막=6785.7M
  LSTM      : 첫값=231.7M  마지막=276.4M
  Theta     : 첫값=345.2M  마지막=367.6M

[Step 3] Long-format: 88 행 (actual + forecast + Ensemble)


ProgrammingError: (pymysql.err.ProgrammingError) nan can not be used with MySQL
[SQL: 
        INSERT INTO `us_revenue_forecast_data`
            (ticker, item, date, period, date_month, data_type, model, value,
             forecast_date, sarima_order, sarima_seasonal_order, sarima_ic_value,
             created_at)
        VALUES
            (%(ticker)s, %(item)s, %(date)s, %(period)s, %(date_month)s, %(data_type)s, %(model)s, %(value)s,
             %(forecast_date)s, %(sarima_order)s, %(sarima_seasonal_order)s, %(sarima_ic_value)s,
             %(created_at)s)
        ON DUPLICATE KEY UPDATE
            value                 = VALUES(value),
            period                = VALUES(period),
            date_month            = VALUES(date_month),
            sarima_order          = VALUES(sarima_order),
            sarima_seasonal_order = VALUES(sarima_seasonal_order),
            sarima_ic_value       = VALUES(sarima_ic_value),
            created_at            = VALUES(created_a

## Cell 10 · 전체 배치 실행 (Override 종목 처리 옵션)

**호영님 요청 5** — Override 종목을 batch 에서 어떻게 다룰지 옵션 선택:

| `BATCH_MODE` | 동작 |
|---|---|
| `"all"` | Override 종목 포함 전체 실행 (override 적용된 데이터로 예측) |
| `"exclude_overrides"` | **Override 종목 제외** — 기본값. Override 종목은 별도로 Cell 9 에서 검토·실행 권장 |
| `"overrides_only"` | Override 종목만 별도 배치 실행 |

`RUN_TICKERS` 가 지정되면 그 안에서 모드 적용. None 이면 `DEFAULT_TICKER_LIST[TICKER_START:TICKER_END]` 사용.

In [ ]:
# ════════════════════════════════════════════════════════════
#  배치 설정 — 여기를 수정
# ════════════════════════════════════════════════════════════

# ── Override 종목 처리 모드 ───────────────────────────────────
BATCH_MODE = "exclude_overrides"   # "all" / "exclude_overrides" / "overrides_only"

# ── 특정 티커 지정 (None 이면 아래 구간/전체 사용) ────────────
RUN_TICKERS  = None                # type: Optional[list]
# RUN_TICKERS = ["STRL", "APH", "FIX"]

# ── 전체 리스트 구간 (RUN_TICKERS=None 일 때) ─────────────────
TICKER_START = 0
TICKER_END   = 2000

# ── 모델 / 항목 / 예측 기간 ───────────────────────────────────
RUN_MODELS  = ["SARIMA", "ETS", "Theta"]    # 또는 ALL_MODELS
RUN_ITEM    = ITEM
RUN_HORIZON = HORIZON
RUN_MIN_OBS = MIN_OBS

# ════════════════════════════════════════════════════════════
#  실행 대상 티커 목록 결정
# ════════════════════════════════════════════════════════════
if RUN_TICKERS is not None:
    base_tickers = RUN_TICKERS
    print(f"[모드] 특정 티커 지정 (총 {len(base_tickers)}개)")
else:
    base_tickers = DEFAULT_TICKER_LIST[TICKER_START:TICKER_END]
    print(f"[모드] 구간 실행: index {TICKER_START} ~ {TICKER_END-1} ({len(base_tickers)}개)")

# Override 처리 모드 적용
if BATCH_MODE == "exclude_overrides":
    tickers  = [t for t in base_tickers if t.upper() not in OVERRIDE_TICKERS]
    excluded = [t for t in base_tickers if t.upper() in OVERRIDE_TICKERS]
    print(f"[Override 처리] EXCLUDE — {len(excluded)} 종목 제외, {len(tickers)} 종목 실행")
    if excluded:
        print(f"   제외된 override 종목: {excluded}")
        print(f"   → Cell 9 에서 개별 실행 권장")
elif BATCH_MODE == "overrides_only":
    tickers = [t for t in base_tickers if t.upper() in OVERRIDE_TICKERS]
    print(f"[Override 처리] OVERRIDES_ONLY — override {len(tickers)} 종목만 실행")
elif BATCH_MODE == "all":
    tickers = base_tickers
    print(f"[Override 처리] ALL — 전체 {len(tickers)} 종목 실행 (override 적용)")
else:
    raise ValueError(f"알 수 없는 BATCH_MODE: {BATCH_MODE}")

total = len(tickers)
if total == 0:
    print("\n[INFO] 실행할 ticker 없음. 종료.")
else:
    # ════════════════════════════════════════════════════════════
    #  배치 실행
    # ════════════════════════════════════════════════════════════
    ensure_table(engine)

    success, skipped, errored, neg_skipped = 0, 0, 0, 0
    skip_list, error_list, neg_skip_list   = [], [], []

    log("BATCH", "=" * 70)
    log("BATCH", f"시작 | {total}개 | mode={BATCH_MODE} | 항목={RUN_ITEM} | 예측={RUN_HORIZON}분기")
    log("BATCH", f"모델 : {RUN_MODELS}  |  예측일: {FORECAST_DATE}  |  min_obs: {RUN_MIN_OBS}")
    log("BATCH", "=" * 70)

    for i, ticker in enumerate(tickers, 1):
        pct = i / total * 100
        log("PROGRESS", f"[{i:>4}/{total}] ({pct:5.1f}%)  >>  {ticker}")

        # Step 1: 데이터 추출 (override 자동 적용)
        try:
            _src_df = fetch_financial_series(engine, ticker, RUN_ITEM, RUN_MIN_OBS)
        except ValueError as e:
            err_msg = str(e)
            if "음수 매출" in err_msg:
                log(ticker, f"[NEG-SKIP] {err_msg}")
                neg_skipped += 1; neg_skip_list.append(ticker)
            else:
                log(ticker, f"[SKIP] {err_msg}")
                skipped += 1; skip_list.append(ticker)
            continue
        except Exception as e:
            log(ticker, f"[SKIP] {e}")
            skipped += 1; skip_list.append(ticker)
            continue

        _y = _src_df.set_index("date")["value"].copy()
        _y.index = pd.DatetimeIndex(_y.index); _y.name = RUN_ITEM
        log(ticker, f"  {len(_y)}분기 | {_y.index[0].date()} ~ {_y.index[-1].date()}")

        # Step 2: 예측
        try:
            _fc_results = forecast_one_ticker(_y, ticker, RUN_HORIZON, RUN_MODELS)
        except Exception as e:
            log(ticker, f"[ERROR] 예측: {e}")
            errored += 1; error_list.append(ticker)
            del _src_df, _y; clear_memory(); continue

        # Step 3: 예측 인덱스
        _fc_index = make_forecast_index(_y.index[-1], RUN_HORIZON)

        # Step 4: Long-format
        try:
            _ldf = build_long_df(
                ticker=ticker, item=RUN_ITEM, src_df=_src_df,
                forecast_results=_fc_results, forecast_index=_fc_index,
                forecast_date=FORECAST_DATE,
            )
        except Exception as e:
            log(ticker, f"[ERROR] Long-format: {e}")
            errored += 1; error_list.append(ticker)
            del _src_df, _y, _fc_results; clear_memory(); continue

        # Step 5: DB 저장
        try:
            save_to_db(engine, _ldf)
            success += 1
        except Exception as e:
            log(ticker, f"[ERROR] DB 저장: {e}")
            errored += 1; error_list.append(ticker)

        del _src_df, _y, _fc_results, _ldf
        clear_memory()

    # ── 요약 ──────────────────────────────────────────────────
    log("BATCH", "=" * 70)
    log("BATCH", f"완료 | 성공: {success}  데이터스킵: {skipped}  음수제외: {neg_skipped}  오류: {errored}  합계: {total}")
    if skip_list:     log("BATCH", f"데이터스킵  : {skip_list[:30]}{'...' if len(skip_list)>30 else ''}")
    if neg_skip_list: log("BATCH", f"음수제외   : {neg_skip_list}")
    if error_list:    log("BATCH", f"오류       : {error_list[:30]}{'...' if len(error_list)>30 else ''}")
    log("BATCH", "=" * 70)

    if BATCH_MODE == "exclude_overrides" and excluded:
        log("BATCH", f"※ 다음 override 종목은 본 배치에서 제외됨 — Cell 9 에서 개별 처리 필요:")
        log("BATCH", f"   {excluded}")


## Cell 11 · 저장 결과 조회

In [ ]:
# ── 오늘 예측된 ticker × 모델별 요약 ───────────────────────
with engine.connect() as conn:
    today_summary = pd.read_sql(
        text(f"""
            SELECT model, COUNT(DISTINCT ticker) AS n_tickers, COUNT(*) AS n_rows
            FROM   `{DEST_TABLE}`
            WHERE  forecast_date = :fd AND data_type = 'forecast'
            GROUP  BY model
            ORDER  BY model
        """),
        conn, params={"fd": FORECAST_DATE},
    )

print(f"[오늘({FORECAST_DATE}) 예측 요약]")
display(today_summary)

# ── 전체 DB 저장 통계 ──────────────────────────────────────
with engine.connect() as conn:
    total_stats = pd.read_sql(
        text(f"""
            SELECT forecast_date,
                   COUNT(DISTINCT ticker) AS n_tickers,
                   COUNT(*) AS n_rows
            FROM   `{DEST_TABLE}`
            GROUP  BY forecast_date
            ORDER  BY forecast_date DESC
            LIMIT  10
        """), conn,
    )

print(f"\n[최근 10일 forecast_date 별 통계]")
display(total_stats)


## Cell 12 · 오염 데이터 삭제 & 재예측 (utility)

특정 ticker / 특정 forecast_date 의 잘못된 데이터를 지울 때 사용.
**삭제는 되돌릴 수 없으므로 신중히** 사용하세요. 기본은 dry-run.

In [ ]:
# ─── 삭제 대상 지정 ──────────────────────────────────────────
DELETE_TICKERS       = ["SIMO"]           # 삭제할 ticker (예: ["STRL"])
DELETE_FORECAST_DATE = FORECAST_DATE      # 삭제할 forecast_date
DRY_RUN              = True               # True 면 실제 삭제 안 함

print(f"[삭제 대상]")
print(f"  ticker(s)     : {DELETE_TICKERS}")
print(f"  forecast_date : {DELETE_FORECAST_DATE}")
print(f"  dry_run       : {DRY_RUN}")

with engine.connect() as conn:
    cnt = pd.read_sql(
        text(f"""
            SELECT ticker, COUNT(*) AS rows_to_delete
            FROM   `{DEST_TABLE}`
            WHERE  ticker IN :tk AND forecast_date = :fd
            GROUP  BY ticker
        """),
        conn,
        params={"tk": tuple(DELETE_TICKERS) if len(DELETE_TICKERS) > 1
                                            else (DELETE_TICKERS[0], DELETE_TICKERS[0]),
                "fd": DELETE_FORECAST_DATE},
    )

print(f"\n[현재 DB 의 해당 행 수]")
display(cnt)

if not DRY_RUN and not cnt.empty:
    with engine.begin() as conn:
        conn.execute(
            text(f"""
                DELETE FROM `{DEST_TABLE}`
                WHERE  ticker IN :tk AND forecast_date = :fd
            """),
            {"tk": tuple(DELETE_TICKERS) if len(DELETE_TICKERS) > 1
                                          else (DELETE_TICKERS[0], DELETE_TICKERS[0]),
             "fd": DELETE_FORECAST_DATE},
        )
    print(f"\n[삭제 완료] 위 행들이 DB 에서 제거됨. Cell 9 또는 Cell 10 으로 재예측하세요.")
elif DRY_RUN:
    print(f"\n[dry-run] DRY_RUN=False 로 변경 후 다시 실행하면 실제 삭제됨.")
else:
    print(f"\n[INFO] 삭제할 행 없음.")
